# ArcLLM Step 2: Config Loading

This notebook walks through everything built in Step 2 — the **config layer** that drives all behavior via TOML files.

**What was built:**
- Global `config.toml` (defaults + module toggles)
- Per-provider TOML files (`providers/anthropic.toml`, `providers/openai.toml`)
- Pydantic config models (`GlobalConfig`, `ProviderConfig`, `ModelMetadata`, etc.)
- `load_global_config()` and `load_provider_config()` loaders
- Path traversal prevention (NIST 800-53 AC-3)

**Why it matters:** Config-driven means agents can switch providers, adjust parameters, and toggle modules without changing code. And validated-on-load means misconfiguration crashes at startup, not during a critical LLM call.

In [ ]:
from arcllm import (
    ArcLLMConfigError,
    DefaultsConfig,
    ModelMetadata,
    ModuleConfig,
    load_global_config,
    load_provider_config,
)

---
## 1. Global Config

The global `config.toml` lives at `src/arcllm/config.toml`. It holds:
- **Defaults**: provider, temperature, max_tokens
- **Module toggles**: which optional features are enabled

Let's load it and explore.

In [ ]:
# Load global config — validated on load, fails fast on any issue
global_config = load_global_config()

In [ ]:
# Explore modules — all disabled by default (opt-in complexity)
for name, _module in global_config.modules.items():
    pass

In [ ]:
# Modules can have extra fields — these are preserved via Pydantic's extra='allow'
# This lets each module define its own settings without changing the config schema
budget = global_config.modules["budget"]

retry = global_config.modules["retry"]

fallback = global_config.modules["fallback"]

rate_limit = global_config.modules["rate_limit"]

In [ ]:
# Full dump to see everything at once

---
## 2. Provider Config

Each provider has its own TOML file under `src/arcllm/providers/`.

A provider config has:
- **`[provider]`** section — connection settings (URL, API key env var, defaults)
- **`[models.*]`** sections — per-model metadata (context window, costs, capabilities)

In [ ]:
# Load Anthropic provider config
anthropic = load_provider_config("anthropic")

In [ ]:
# Explore model metadata
for model_name, meta in anthropic.models.items():
    pass

In [ ]:
# Compare with OpenAI provider
openai = load_provider_config("openai")

for model_name, meta in openai.models.items():
    pass

### Comparing Providers

Because everything is normalized to the same types, you can compare across providers easily.

In [ ]:
# Side-by-side model comparison

for provider_name, provider_config in [("anthropic", anthropic), ("openai", openai)]:
    for model_name, meta in provider_config.models.items():
        tools_str = "yes" if meta.supports_tools else "no"

---
## 3. Using Model Metadata

Model metadata isn't just informational — agents use it to make decisions:
- Can this model use tools? Check `supports_tools`
- Can I send images? Check `supports_vision` and `input_modalities`
- How much will this cost? Check `cost_*_per_1m`
- Will my prompt fit? Check `context_window`

In [ ]:
# Simulate: pick the cheapest model that supports tools
all_models = []
for provider_name, config in [("anthropic", anthropic), ("openai", openai)]:
    for model_name, meta in config.models.items():
        all_models.append((provider_name, model_name, meta))

tool_capable = [(p, m, meta) for p, m, meta in all_models if meta.supports_tools]
cheapest = min(tool_capable, key=lambda x: x[2].cost_input_per_1m)

In [ ]:
# Simulate: check if a prompt fits
sonnet = anthropic.models["claude-sonnet-4-20250514"]

estimated_tokens = 150_000  # big prompt
if estimated_tokens < sonnet.context_window:
    remaining = sonnet.context_window - estimated_tokens
else:
    pass

In [ ]:
# Simulate: estimate cost for a batch of calls
n_calls = 1000
avg_input = 2000  # tokens per call
avg_output = 500  # tokens per call

model_meta = anthropic.models["claude-sonnet-4-20250514"]
total_input = n_calls * avg_input
total_output = n_calls * avg_output

input_cost = (total_input / 1_000_000) * model_meta.cost_input_per_1m
output_cost = (total_output / 1_000_000) * model_meta.cost_output_per_1m
total_cost = input_cost + output_cost

---
## 4. Security: Path Traversal Prevention

Because `load_provider_config()` takes a string and turns it into a file path, we MUST validate it. Otherwise an attacker could pass `../../etc/passwd` as a provider name.

**Decision D-035**: Strict regex validation — `^[a-z][a-z0-9\-]*$`, max 64 characters.

This satisfies NIST 800-53 AC-3 (access enforcement) and OWASP A01 (broken access control).

In [ ]:
# Path traversal attack — BLOCKED
try:
    load_provider_config("../../etc/passwd")
except ArcLLMConfigError:
    pass

In [ ]:
# Various attack vectors — all blocked
attacks = [
    ("../evil", "parent directory traversal"),
    ("pro/vider", "slash in name"),
    (".hidden", "dot-prefixed"),
    ("a@b", "special characters"),
    ("a b", "spaces"),
    ("", "empty string"),
    ("Anthropic", "uppercase (case-sensitive match)"),
    ("a" * 65, "name too long (>64 chars)"),
]

for attack, _description in attacks:
    try:
        load_provider_config(attack)
    except ArcLLMConfigError:
        display_name = attack if len(attack) <= 30 else attack[:27] + "..."

In [ ]:
# Valid provider names that DO work
from arcllm.config import _validate_provider_name

valid_names = ["anthropic", "openai", "ollama", "my-custom-provider", "o1", "gpt4"]
for name in valid_names:
    try:
        _validate_provider_name(name)
    except ArcLLMConfigError:
        pass

---
## 5. Error Handling: Fail-Fast on Bad Config

Config errors are caught at load time, not during an LLM call. This is critical for unattended agents — you want to crash at startup, not 3 hours into a batch job.

In [ ]:
# Missing provider file
try:
    load_provider_config("nonexistent")
except ArcLLMConfigError:
    pass

In [ ]:
# Malformed TOML (simulated with unittest.mock)
import tempfile
from pathlib import Path
from unittest.mock import patch

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    # Write a broken TOML file
    bad_toml = tmp_path / "config.toml"
    bad_toml.write_text("[unclosed section")

    with patch("arcllm.config._get_config_dir", return_value=tmp_path):
        try:
            load_global_config()
        except ArcLLMConfigError:
            pass

In [ ]:
# Wrong types in TOML (temperature should be float, not string)
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    bad_types = tmp_path / "config.toml"
    bad_types.write_text('[defaults]\nprovider = "ok"\ntemperature = "not-a-float"\n')

    with patch("arcllm.config._get_config_dir", return_value=tmp_path):
        try:
            load_global_config()
        except ArcLLMConfigError:
            pass

In [ ]:
# All config errors are ArcLLMConfigError -> ArcLLMError
# So you can catch them at whatever granularity you need
try:
    load_provider_config("nonexistent")
except ArcLLMConfigError:
    pass

try:
    load_provider_config("nonexistent")
except Exception:
    pass

---
## 6. Config Models in Detail

Let's look at each Pydantic config model and how they compose.

In [ ]:
# DefaultsConfig — has sensible defaults for every field
# You can create one with no arguments and get reasonable values
defaults_empty = DefaultsConfig()

In [ ]:
# ModuleConfig — extra='allow' is the magic
# It accepts any extra fields without complaining
basic = ModuleConfig(enabled=True)

# With extra fields (module-specific settings)
budget_mod = ModuleConfig(enabled=True, monthly_limit_usd=1000.0, alert_threshold=0.8)

# The extra fields show up in model_dump too

In [ ]:
# ModelMetadata — everything you need to know about a model
# All fields are required (no guessing in production)
from pydantic import ValidationError

try:
    ModelMetadata(context_window=100000)  # missing required fields
except ValidationError as e:
    for _err in e.errors():
        pass

In [ ]:
# ProviderSettings — connection info for making API calls
# Note: api_key_env is the ENV VAR NAME, never the actual key
ps = anthropic.provider

---
## 7. Config Discovery

Config files are discovered **package-relative** via `Path(__file__).parent`.

This means configs work in both:
- Development mode (`pip install -e .`)
- Installed mode (`pip install arcllm`)

In [ ]:
from arcllm.config import _get_config_dir

config_dir = _get_config_dir()
for f in sorted(config_dir.rglob("*.toml")):
    pass

In [ ]:
# Let's read the raw TOML to see what the loader parses
import tomllib

config_path = config_dir / "config.toml"
with open(config_path, "rb") as f:
    raw_data = tomllib.load(f)

In [ ]:
# And a provider TOML
provider_path = config_dir / "providers" / "anthropic.toml"
with open(provider_path, "rb") as f:
    raw_provider = tomllib.load(f)

---
## 8. Switching Providers

One of the key benefits of config-driven design: switching providers is just loading a different TOML.

In [ ]:
# Load both and compare
providers = {}
for name in ["anthropic", "openai"]:
    providers[name] = load_provider_config(name)

In [ ]:
# Pattern: agent selects provider based on global defaults
global_cfg = load_global_config()
default_provider = global_cfg.defaults.provider

provider_cfg = load_provider_config(default_provider)
default_model = provider_cfg.provider.default_model
model_meta = provider_cfg.models[default_model]

---
## 9. Override Chain Preview

The merge strategy (Decision D-033) is: **args > provider TOML > global TOML**

This isn't implemented yet (it will be in `load_model()`), but here's how it will work conceptually.

In [ ]:
# Preview: how overrides will chain
global_temp = global_cfg.defaults.temperature  # 0.7 (from config.toml)
provider_temp = provider_cfg.provider.default_temperature  # 0.7 (from anthropic.toml)
user_temp = 0.0  # agent passes temperature=0 for deterministic output

---
## 10. Putting It Together: What an Agent Startup Looks Like

Here's the pattern agents will use once everything is wired up.

In [ ]:
def agent_startup_preview():
    """What agent initialization will look like."""
    # 1. Load global config
    global_cfg = load_global_config()

    # 2. Load provider config
    provider_name = global_cfg.defaults.provider
    provider_cfg = load_provider_config(provider_name)

    # 3. Select model
    model_name = provider_cfg.provider.default_model
    provider_cfg.models[model_name]

    # 4. Check API key exists (security: env vars only)

    # 5. Ready (Step 6 will return an LLMProvider here)


agent_startup_preview()

---
## Summary

Step 2 built the **config layer**:

```
config.toml          ->  GlobalConfig (defaults + module toggles)
providers/*.toml      ->  ProviderConfig (connection settings + model metadata)
config.py            ->  Pydantic models + loaders + path traversal prevention
```

**Key design decisions:**
- TOML format (stdlib `tomllib`, zero dependencies)
- Pydantic validation on load (fail-fast for agents)
- Package-relative file discovery (`Path(__file__).parent`)
- Simple override chain: args > provider > global
- Strict regex on provider names (path traversal prevention)
- `extra='allow'` on ModuleConfig (extensible without schema changes)
- API keys referenced by env var name, never stored in config